In [1]:
%cd /home/maia-user/NeuroCBIR/
!ls

/home/maia-user/NeuroCBIR
cbir	    data	main.ipynb  preprocessing  seg_preparation.ipynb
CBIR.ipynb  evaluation	model	    README.md	   utils


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
# import torch
# from model.autoencoder import Conv3DSparseAutoencoder
# from torchsummary import summary

# # Device setup
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Input size: [160, 176, 208]
# input_size = [64, 80, 48]

# # Initialize model and move it to the correct device
# autoencoder = Conv3DSparseAutoencoder().to(device)

# # Run summary — torchsummary will match the device of the model
# _=summary(autoencoder, (1, *input_size))

In [4]:
import sys
print(sys.executable)


/usr/bin/python3


In [5]:
import torch
import torch.nn as nn
import os
import pandas as pd
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from IPython.display import clear_output

seed = 42

In [8]:
from preprocessing.load_dataset import list_files_with_extension, BrainMRIDataset, get_label

# Loading MRI  paths
dataset_path = r"/home/maia-user/Dataset/OASIS3_NPY_UINT/"
file_paths, file_names = list_files_with_extension(dataset_path, extension=".npy")
raw_image_paths = np.array([os.path.join(dataset_path, file_path, file_name) for file_path, file_name in zip(file_paths, file_names)])
raw_image_ids = np.array([file_path.split('.')[0] for file_path in file_names])

# Load file with labels
labels_path = r"/home/maia-user/Dataset/OASIS3/OASIS3_UDSb4_cdr.xlsx"
labels_df = pd.read_excel(labels_path, sheet_name= 'Labels')

# Some recordings might be removed, i.e. nan labels
raw_labels = np.array([get_label(image_id, labels_df) for image_id in raw_image_ids]).astype('float32')
raw_ages = np.array([get_label(image_id, labels_df, column='age_at_visit') for image_id in raw_image_ids]).astype('float32')
raw_ids = np.array([get_label(image_id, labels_df, column='OASISID') for image_id in raw_image_ids])

# Filtering out undesired cases
labels = raw_labels[~np.isnan(raw_labels)]
image_paths = raw_image_paths[~np.isnan(raw_labels)]
ages = raw_ages[~np.isnan(raw_labels)]
ids = raw_ids[~np.isnan(raw_labels)]

print(raw_labels.shape, labels.shape)

/home/maia-user/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(2681,) (2680,)


In [9]:
from preprocessing.split_dataset import stratified_patient_split

# Longitudinal splitting dataset into training and testing sets
train_set, val_set, test_set = stratified_patient_split(image_paths, labels, ages, ids)

# Print dataset sizes
print(f"Training samples: {len(train_set['image_paths'])} - IR: {np.sum(train_set['labels'])/len(train_set['image_paths']):.2f}")
print(f"Validation samples: {len(val_set['image_paths'])} - IR: {np.sum(val_set['labels'])/len(val_set['image_paths']):.2f}")
print(f"Testing samples: {len(test_set['image_paths'])} - IR: {np.sum(test_set['labels'])/len(val_set['image_paths']):.2f}")

Training samples: 1852 - IR: 0.20
Validation samples: 372 - IR: 0.21
Testing samples: 456 - IR: 0.26


In [10]:
from preprocessing.load_dataset import StratifiedBatchSampler
import torchio as tio

batch_size = 24  # Make sure this is even!

augmentation_transforms = tio.Compose([
                                        # tio.RandomAffine(
                                        #     scales=(0.9, 1.05),        # Random scaling
                                        #     degrees=5,               # Random rotation in degrees
                                        #     translation=5            # Random translation in mm
                                        # ),
                                        tio.RandomNoise(mean=0.0, std=0.02),
                                        # tio.RandomFlip(axes=('LR',)),  # Random left-right flip
                                    ])

# Preparing the dataset to feed the network
train_dataset = BrainMRIDataset(train_set['image_paths'], train_set['ages'], train_set['labels'], 
                                transform=None, transform_age=3, cache=False, sparse_path='./data/common_bounding_boxes.json')
train_sampler = StratifiedBatchSampler(train_dataset, batch_size)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, pin_memory=True, num_workers=1)

val_dataset = BrainMRIDataset(val_set['image_paths'], val_set['ages'], val_set['labels'], cache=False, sparse_path='./data/common_bounding_boxes.json')
val_sampler = StratifiedBatchSampler(val_dataset, batch_size)
val_loader = DataLoader(val_dataset, batch_sampler=val_sampler, num_workers=1)

test_dataset = BrainMRIDataset(test_set['image_paths'], test_set['ages'], test_set['labels'], cache=False, sparse_path='./data/common_bounding_boxes.json')
test_sampler = StratifiedBatchSampler(test_dataset, batch_size)
test_loader = DataLoader(test_dataset, batch_sampler=test_sampler, num_workers=1)

In [ ]:
from model.autoencoder import Conv3DSparseAutoencoder

# Load dataset features
dataset_feats = np.load("./data/dataset_feats_64_32_16_8_64_NR.npy", allow_pickle=True)

# Load model
# Config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bottleneck_dim=128
autoencoder = Conv3DSparseAutoencoder(bottleneck_dim=bottleneck_dim).to(device)
pretrained_param = torch.load('./data/pretrained_models/CP_20250424_150510.pth')
autoencoder.load_state_dict(pretrained_param)



In [ ]:
sample_1 = dataset[699]  # Any sample

autoencoder.eval()

with torch.no_grad():
    subc_struc_ind = 0
    input_img = sample_1["input"].unsqueeze(0).to(device, dtype=torch.float32)[0:1, subc_struc_ind:subc_struc_ind+1]
    z, reconstructed = autoencoder(input_img)

# Convert to NumPy
original_np = input_img.squeeze().cpu().numpy()
reconstructed_np = reconstructed.squeeze().cpu().numpy()

# Compute mid-slices
pz = original_np.shape[0] // 2  # Axial
py = original_np.shape[1] // 2  # Coronal
px = original_np.shape[2] // 2  # Sagittal

# Extract slices
slices = {
    "Axial (Z)":      (original_np[pz, :, :], reconstructed_np[pz, :, :]),
    "Coronal (Y)":    (original_np[:, py, :], reconstructed_np[:, py, :]),
    "Sagittal (X)":   (original_np[:, :, px], reconstructed_np[:, :, px]),
}

# Plotting
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, (title, (orig, recon)) in enumerate(slices.items()):
    axes[0, i].imshow(orig, cmap='gray')
    axes[0, i].set_title(f'Original - {title}')
    axes[0, i].axis('off')

    axes[1, i].imshow(recon, cmap='gray')
    axes[1, i].set_title(f'Reconstructed - {title}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

array([{'subject_id': np.str_('OAS30647'), 'record_id': 'OAS30647_MR_d0232', 'subc_str': 'Hippocampus (lh)', 'label': 0.0, 'age': 55.16561508178711, 'features': array([ 2.7699037 ,  0.95168304, -1.5440829 , -0.44443378,  1.4007516 ,
              -1.070506  ,  0.20305486, -1.078201  ,  0.4742176 ,  0.22284068,
              -0.36030465,  0.9107693 , -0.5680814 , -0.87072974,  1.170667  ,
              -0.06766762, -0.9460955 ,  2.1330466 , -0.12332818, -1.4970486 ,
              -0.49326512, -0.0558378 , -0.09106684,  0.21269047,  0.33652267,
               1.6974072 , -0.11401122, -0.36504424,  0.612497  ,  0.923581  ,
               0.64454573, -0.64217585,  1.0079165 , -0.65684795,  1.1859313 ,
              -0.10449945, -1.0998945 ,  0.21244852, -0.1435305 , -0.45535114,
               2.060122  , -1.2998637 , -0.50252336,  0.08346002, -0.28984484,
              -0.79258925, -0.8089439 , -0.36918318, -0.19052982, -1.5731561 ,
               1.5641491 , -0.58094877,  0.26719588,  1.